<a href="https://colab.research.google.com/github/Azzesh/Web_CC1/blob/main/AnimateDiff_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
# @title 1. Install & Setup
!pip install diffusers transformers accelerate opencv-python-headless tqdm mediapy

import torch
import numpy as np
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
from tqdm.notebook import tqdm
import cv2
import mediapy as media
import os
from IPython.display import Video

# Check for GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Using device: {device}")

🚀 Using device: cuda


In [32]:
# @title 2. Set Your Prompt & Parameters

# --- Your Prompt ---
prompt = "A beautiful scenecry of an old home with a dog sitting in front "  # @param {type:"string"}
negative_prompt = "ugly, tiling, poorly drawn, out of frame, disfigured, deformed" # @param {type:"string"}

# --- Technical Settings ---
guidance_scale = 7.5 # @param {type:"number"}
height = 704          # @param {type:"slider", min:256, max:768, step:64}
width = 640           # @param {type:"slider", min:256, max:768, step:64}
seed = 100             # @param {type:"integer"}
num_inference_steps = 225 # @param {type:"integer"}

print(f"✨ Generating video for prompt: '{prompt}'")

✨ Generating video for prompt: 'A beautiful scenecry of an old home with a dog sitting in front '


In [ ]:
# @title 5. Stretch Video to 15 Seconds with Smooth Interpolation
import subprocess
import os

input_video = "diffusion_process.mp4"
output_video = "diffusion_process_15sec.mp4"

# Step 1: Get original duration and frame rate
probe = subprocess.run(
    ['ffprobe', '-v', 'error', '-select_streams', 'v:0',
     '-show_entries', 'stream=duration,r_frame_rate', '-of', 'default=noprint_wrappers=1', input_video],
    capture_output=True, text=True
)
lines = probe.stdout.strip().split('\n')
duration = float([l for l in lines if 'duration' in l][0].split('=')[1])
fps_str = [l for l in lines if 'r_frame_rate' in l][0].split('=')[1]
num, den = map(int, fps_str.split('/'))
fps = num / den if den != 0 else float(fps_str)

print(f"Original: {duration:.2f} sec, {fps:.2f} fps, {int(duration*fps)} frames")

# Desired new duration = 15 sec
new_duration = 15.0
speed_factor = new_duration / duration  # >1 means slow down

# Step 2: Slow down video to 15 sec (keeps original frames, lowers fps)
slow_video = "temp_slow.mp4"
subprocess.run([
    'ffmpeg', '-i', input_video,
    '-filter:v', f'setpts={speed_factor}*PTS',
    '-r', str(fps / speed_factor),   # new fps = original fps / speed_factor
    '-an',  # no audio
    '-y', slow_video
], check=True, capture_output=True)

# Step 3: Interpolate to restore 15 fps (creates new frames)
subprocess.run([
    'ffmpeg', '-i', slow_video,
    '-filter:v', f'minterpolate=fps=15:mi_mode=mci:mc_mode=aobmc:me_mode=bidir',
    '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-crf', '18',
    '-y', output_video
], check=True, capture_output=True)

# Clean up temporary file
os.remove(slow_video)

print(f"✅ New video created: {output_video}")
print(f"Duration: 15 seconds at 15 fps → {15*15} frames (interpolated smoothly)")

Original: 66.67 sec, 15.00 fps, 1000 frames


In [ ]:
# @title 3. Build the Frame-by-Frame Video (Manual Loop - Reliable)

# 1. Load the Model
model_id = "runwayml/stable-diffusion-v1-5"
pipe = StableDiffusionPipeline.from_pretrained(
    model_id, torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
print("✅ Model loaded.")

# 2. Prepare for manual denoising
output_dir = "frames"
os.makedirs(output_dir, exist_ok=True)

# Encode the prompt
text_embeddings = pipe.encode_prompt(
    prompt, device, 1, True, negative_prompt
)
# In recent diffusers, encode_prompt returns a tuple (prompt_embeds, negative_prompt_embeds)
if isinstance(text_embeddings, tuple):
    prompt_embeds, negative_prompt_embeds = text_embeddings
    text_embeddings = torch.cat([negative_prompt_embeds, prompt_embeds])
else:
    # Old version: just a single tensor
    text_embeddings = text_embeddings

# Set up latents
generator = torch.Generator(device=device).manual_seed(seed)
latents = torch.randn(
    (1, pipe.unet.config.in_channels, height // 8, width // 8),
    generator=generator,
    device=device,
    dtype=pipe.unet.dtype,
)
pipe.scheduler.set_timesteps(num_inference_steps)
timesteps = pipe.scheduler.timesteps

print(f"🎨 Generating {len(timesteps)} frames...")

# 3. Denoising loop
for i, t in enumerate(tqdm(timesteps, desc="Denoising")):
    # Expand latents for classifier-free guidance
    latent_model_input = torch.cat([latents] * 2)
    latent_model_input = pipe.scheduler.scale_model_input(latent_model_input, t)

    # Predict noise
    with torch.no_grad():
        noise_pred = pipe.unet(
            latent_model_input, t, encoder_hidden_states=text_embeddings
        ).sample

    # Perform guidance
    noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
    noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)

    # Update latents
    latents = pipe.scheduler.step(noise_pred, t, latents).prev_sample

    # Decode and save frame
    with torch.no_grad():
        image = pipe.vae.decode(latents / pipe.vae.config.scaling_factor, return_dict=False)[0]
    image = (image / 2 + 0.5).clamp(0, 1)
    image = image.cpu().permute(0, 2, 3, 1).float().numpy()
    image = (image[0] * 255).round().astype("uint8")
    frame_path = os.path.join(output_dir, f"frame_{i:04d}.png")
    cv2.imwrite(frame_path, cv2.cvtColor(image, cv2.COLOR_RGB2BGR))

print("✅ Generation complete.")

# 4. Create Video from Frames
frame_files = sorted([os.path.join(output_dir, f) for f in os.listdir(output_dir) if f.endswith('.png')])
if frame_files:
    first_frame = cv2.imread(frame_files[0])
    h, w, _ = first_frame.shape
    video_name = "diffusion_process.mp4"
    out = cv2.VideoWriter(video_name, cv2.VideoWriter_fourcc(*'mp4v'), 15, (w, h))
    for frame_file in frame_files:
        frame = cv2.imread(frame_file)
        out.write(frame)
    out.release()
    print(f"🎬 Video saved as '{video_name}' with {len(frame_files)} frames.")
else:
    print("No frames were generated.")

In [ ]:
# @title 6. Watch the 15‑Second Video
from IPython.display import Video
Video("diffusion_process_15sec.mp4", embed=True, width=512)

In [31]:
from google.colab import files
files.download("diffusion_process.mp4")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>